<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week09_Expert_Systems_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 9 Practical (PH):** Expert systems — rules, inference engines, certainty factors and a help-desk chatbot

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO8

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — The inference engine | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part B — A crop-disease advisor (cassava) — *teaching toy, not agronomic advice* | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Build *your own* expert system (knowledge engineering task) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part D — A help-desk chatbot (CCMAS lab: *staff/student chat services*) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | name the **components** of an expert system and map them to code | Understand | LO8 |
| 2 | implement **forward chaining** with **conflict resolution** | Apply | LO8 |
| 3 | implement **backward chaining** and compare the number of questions asked | Apply / Analyse | LO8 |
| 4 | handle uncertainty with **certainty factors** and give **explanations** (*how?*) | Apply | LO8 |
| 5 | develop a small expert system for a domain of your choice (knowledge engineering) | Create | LO8 |

**Architecture.** *Knowledge base* (IF–THEN rules) · *working memory* (facts about the current case) · *inference engine* (forward / backward chaining + conflict resolution) · *explanation facility* ("how did you conclude that?") · *user interface* · *knowledge-acquisition module*. 

**Certainty factors (MYCIN-style).** A rule $R$: IF $E_1\wedge\dots\wedge E_k$ THEN $H$ with strength $\mathrm{cf}_R\in[0,1]$ gives
$$\mathrm{cf}(H)=\mathrm{cf}_R\cdot\min_i \mathrm{cf}(E_i).$$
Two independent rules supporting the same $H$ with $c_1,c_2\ge 0$ combine as $\;c_1\oplus c_2=c_1+c_2(1-c_1)$ (evidence accumulates but never exceeds 1).

> ⚠️ **A certainty factor is not a probability.** $\mathrm{cf}=0.8$ means "the evidence supports this strongly", not "true 80% of the time". CFs do not obey Bayes' rule and the combination formula is a convention that works best when pieces of evidence are roughly independent. Never *add* certainty factors ($0.72+0.48=1.2$ is meaningless); always combine with $\oplus$.

**Worked check (class note, Example 9.3).** Facts: mosaic 1.0, distortion 0.9, stunting 0.8. $R_1$ gives $0.8\cdot\min(1.0,0.9)=0.72$; $R_2$ gives $0.6\cdot\min(1.0,0.8)=0.48$; combined $0.72\oplus0.48=0.854$; the chained advice rule ($0.9$) gives $0.769$.

**Do Now (3 min).** Pick a domain where a human expert is scarce in your community (e.g. crop disease, network faults, triage). Why are rules a good (or bad) fit?

## Part A — The inference engine
We implement our own engine (no external library to break). A **rule** has a name, a list of condition facts, one conclusion, a certainty factor and a priority. **Working memory** maps *fact → certainty factor*.

**Conflict resolution.** When several rules can fire, choose one: `first` (rule order), `priority` (highest number), `specificity` (most conditions = most specific).

In [ ]:
#@title ⚙️ Rules, certainty-factor algebra and conflict resolution (TODO: two small functions)
# dataclass builds the Rule class automatically
from dataclasses import dataclass, field
# Type hints for lists
from typing import List

# frozen=True makes rules immutable (safe to store in sets)
@dataclass(frozen=True)
# A production rule: IF conditions THEN conclusion
class Rule:
    # Rule name
    name: str
    # The IF part: a tuple of required facts
    if_: tuple
    # The THEN part: the fact it concludes
    then: str
    # Certainty factor of the rule (0 to 1)
    cf: float = 1.0
    # Priority used by conflict resolution
    priority: int = 0

# Evidence weaker than this is ignored (as in MYCIN)
THRESHOLD = 0.2                                # evidence weaker than this is ignored (as in MYCIN)

def combine_cf(c1, c2):
    """TODO(Student): combine two positive certainty factors: c1 + c2 (1 − c1)."""
    # Placeholder: returns the first certainty; replace with the combination formula
    return c1

def pick_rule(conflict_set, strategy="first"):
    """TODO(Student): choose ONE rule from a non-empty list.
       'first' → first in list; 'priority' → highest .priority; 'specificity' → most conditions (ties: first)."""
    # Placeholder: always picks the first rule in the conflict set
    return conflict_set[0]

In [ ]:
#@title ✅ Auto-check for the two functions
# Three rules that all match, to test the strategies
r_a, r_b, r_c = Rule("a", ("x",), "p", priority=1), Rule("b", ("x", "y"), "q", priority=5), Rule("c", ("x", "y", "z"), "r", priority=3)
check("combine_cf(0.8, 0.6) = 0.92",                lambda: abs(combine_cf(0.8, 0.6) - 0.92) < 1e-9)
check("combine_cf is symmetric and ≤ 1",            lambda: abs(combine_cf(0.3, 0.9) - combine_cf(0.9, 0.3)) < 1e-9 and combine_cf(0.99, 0.99) <= 1)
check("'first' picks rule a",                       lambda: pick_rule([r_a, r_b, r_c], "first").name == "a")
check("'priority' picks rule b (5)",                lambda: pick_rule([r_a, r_b, r_c], "priority").name == "b")
check("'specificity' picks rule c (3 conditions)",  lambda: pick_rule([r_a, r_b, r_c], "specificity").name == "c")

In [ ]:
#@title 🧠 Forward chaining, backward chaining and explanation
# The inference engine
class Engine:
    # Store the rule base
    def __init__(self, rules): self.rules = list(rules)

    # ---------- forward chaining (data-driven) ----------------------------------------------------------
    # Forward chaining: from facts to conclusions
    def forward(self, facts, strategy="first"):
        # Working memory, fired rules, trace, and the explanation (HOW) table
        wm = dict(facts); fired = set(); trace = []; how = {f: ("given", ()) for f in facts}
        # Repeat until no rule can fire
        while True:
            # Conflict set: unfired rules whose conditions all hold above the threshold
            conflict = [r for r in self.rules if r.name not in fired and all(wm.get(c, 0) >= THRESHOLD for c in r.if_)]
            # Stop when nothing can fire
            if not conflict: break
            # Choose a rule and mark it fired (refraction)
            r = pick_rule(conflict, strategy); fired.add(r.name)                    # refraction: each rule fires once
            # Certainty = rule CF times the weakest condition (the minimum)
            cf = r.cf * min(wm[c] for c in r.if_)
            # Add the conclusion, combining certainties if it already exists
            wm[r.then] = combine_cf(wm[r.then], cf) if r.then in wm else cf
            # Record how the conclusion was reached for later explanation
            how[r.then] = (r.name, r.if_) if r.then not in how or how[r.then][0] == "given" else (how[r.then][0] + "+" + r.name, how[r.then][1] + r.if_)
            # Add to the trace
            trace.append((r.name, r.then, round(wm[r.then], 3)))
        # Keep the explanation table
        self.how = how
        # Return final facts and the trace
        return wm, trace

    # ---------- backward chaining (goal-driven) -------------------------------------------------------------
    def backward(self, goal, ask, known=None, asked=None):
        """cf of `goal`; asks the user (via `ask(fact) → cf`) only for facts no rule can conclude."""
        # Backward chaining: start from the goal and ask only needed questions
        known = {} if known is None else known; asked = [] if asked is None else asked
        # If already known, reuse it
        if goal in known: return known[goal], asked
        # Rules that can conclude this goal
        rules = [r for r in self.rules if r.then == goal]
        # No rule concludes it: it is a primitive observation, ask the user
        if not rules:                                                                # askable: a primitive observation
            # Ask and remember the answer
            known[goal] = ask(goal); asked.append(goal); return known[goal], asked
        # Total certainty for the goal
        total = 0.0
        # Try each rule that concludes the goal
        for r in rules:
            # Certainties of the conditions
            cfs = []
            # Prove each condition recursively
            for c in r.if_:
                # Store its certainty
                v, _ = self.backward(c, ask, known, asked); cfs.append(v)
                # Weak condition: this rule cannot succeed, stop asking
                if v < THRESHOLD: break                                              # this rule cannot succeed → stop asking
            # If all conditions hold, combine the rule's certainty with the total
            if len(cfs) == len(r.if_) and min(cfs) >= THRESHOLD: total = combine_cf(total, r.cf * min(cfs))
        # Remember the result
        known[goal] = total
        # Return certainty and the questions asked
        return total, asked

    def explain(self, fact, indent=0):
        """Explanation facility: HOW was `fact` obtained?"""
        # Look up how a fact was concluded
        src, conds = self.how.get(fact, ("unknown", ()))
        # Print the fact and its source
        print("  " * indent + f"{fact}  ← {src}")
        # Explain each condition (indented)
        for c in dict.fromkeys(conds): self.explain(c, indent + 1)

## Part B — A crop-disease advisor (cassava) — *teaching toy, not agronomic advice*
Knowledge engineering means turning an expert's heuristics into rules. The rules below are simplified from common descriptions of cassava mosaic disease (CMD), bacterial blight (CBB), brown streak disease (CBSD) and mealybug damage; a real system must be built with **extension officers / plant pathologists** and validated.

In [ ]:
#@title 🌱 The knowledge base
# The expert system for cassava diseases
RULES = [
  Rule("R1", ("leaf_mosaic", "leaf_distortion"), "cmd", 0.8),
  Rule("R2", ("leaf_mosaic", "stunted_growth"), "cmd", 0.6),
  Rule("R3", ("angular_leaf_spots", "wilting"), "cbb", 0.7),
  Rule("R4", ("gum_exudate",), "cbb", 0.5),
  Rule("R5", ("yellow_vein_patches", "brown_stem_streaks"), "cbsd", 0.8),
  Rule("R6", ("root_necrosis",), "cbsd", 0.6),
  Rule("R7", ("bunchy_tip", "white_mealybugs_visible"), "mealybug", 0.9),
  Rule("R8", ("cmd",), "advise_clean_cuttings_and_resistant_varieties", 0.9),
  Rule("R9", ("cbb",), "advise_remove_infected_plants_and_disinfect_tools", 0.8),
  Rule("R10", ("cbsd",), "advise_use_certified_cuttings_and_harvest_early", 0.8),
  Rule("R11", ("mealybug",), "advise_consult_extension_officer_for_control", 0.9),
]
# Build the engine
eng = Engine(RULES)
# Case 1: symptoms with certainty values
case1 = {"leaf_mosaic": 1.0, "leaf_distortion": 0.9, "stunted_growth": 0.8}
# Run forward chaining
wm, trace = eng.forward(case1)
# Print the rules that fired
print("Forward chaining trace:"); [print("  fired", t) for t in trace]
# Ask HOW the advice was derived
print("\nHOW was the advice reached?"); eng.explain("advise_clean_cuttings_and_resistant_varieties")
check("CMD cf = 0.72 ⊕ 0.48 = 0.8544 (two rules accumulate)", lambda: abs(wm["cmd"] - combine_cf(0.8 * 0.9, 0.6 * 0.8)) < 1e-9)
check("chain of inference: advice derived from the diagnosis", lambda: "advise_clean_cuttings_and_resistant_varieties" in wm)
check("unrelated diagnoses are NOT derived",                    lambda: "cbb" not in wm and "cbsd" not in wm)

### Forward vs backward chaining — how many questions do we ask?
*Forward* chaining needs **all** observations up front. *Backward* chaining starts from a hypothesis ("is it CBB?") and asks **only** what is relevant. We simulate the farmer with an answer table and count questions.

In [ ]:
#@title ❓ Backward chaining asks fewer questions
# All possible symptom questions
ALL_SYMPTOMS = ["leaf_mosaic", "leaf_distortion", "stunted_growth", "angular_leaf_spots", "wilting", "gum_exudate",
                "yellow_vein_patches", "brown_stem_streaks", "root_necrosis", "bunchy_tip", "white_mealybugs_visible"]
# Farmer's answers (others default to 0)
farmer = {s: 0.0 for s in ALL_SYMPTOMS}; farmer.update({"angular_leaf_spots": 1.0, "wilting": 0.9, "gum_exudate": 0.7})
# The ask function answers from the farmer's data
ask = lambda f: farmer.get(f, 0.0)
# Backward-chain the goal CBB
cf_cbb, asked_cbb = eng.backward("cbb", ask)
# Backward-chain the advice goal
cf_adv, asked_adv = eng.backward("advise_remove_infected_plants_and_disinfect_tools", ask)
# Print certainty and questions asked
print(f"P(CBB) cf = {cf_cbb:.3f}; questions asked = {len(asked_cbb)} of {len(ALL_SYMPTOMS)}:", asked_cbb)
# Print the advice certainty
print(f"advice cf = {cf_adv:.3f}")
# Try another goal (cassava mosaic disease)
cf_cmd, asked_cmd = eng.backward("cmd", ask); print(f"CMD cf = {cf_cmd:.3f}; questions asked = {len(asked_cmd)}:", asked_cmd)
check("backward chaining asks fewer questions than forward chaining needs (11)", lambda: len(asked_cbb) < len(ALL_SYMPTOMS))
check("CBB cf = 0.7·0.9 ⊕ 0.5·0.7 = 0.8195", lambda: abs(cf_cbb - combine_cf(0.7 * 0.9, 0.5 * 0.7)) < 1e-9)
check("CMD is not supported (cf = 0)", lambda: cf_cmd == 0.0)

### Conflict resolution matters for the *order* of firing (and sometimes the outcome)
Three rules can fire at once. Compare the traces under different strategies.

In [ ]:
#@title 🔀 Same facts, different strategy
# Facts that trigger two rules at once
facts = {"gum_exudate": 1.0, "yellow_vein_patches": 1.0, "brown_stem_streaks": 1.0}   # R4 (1 condition) is listed before R5 (2 conditions)
# Compare the two conflict-resolution strategies
for strat in ["first", "specificity"]:
    # Print the order in which rules fired
    _, tr = eng.forward(facts, strat); print(f"{strat:<12}", [t[0] for t in tr])
check("different strategies give different firing orders", lambda: [t[0] for t in eng.forward(facts, "first")[1]] != [t[0] for t in eng.forward(facts, "specificity")[1]])
check("…but the same final conclusions (confluence for monotone rules)", lambda: set(eng.forward(facts, "first")[0]) == set(eng.forward(facts, "specificity")[0]))

## Part C — Build *your own* expert system (knowledge engineering task)
Choose one domain: **device troubleshooting**, **network diagnosis**, **course advising**, **health-centre triage (non-clinical demo)**, or **loan pre-screening**. Write **≥ 6 rules** with certainty factors, at least **one two-step chain** (a conclusion used as a condition), and run it on **two cases** that produce different conclusions. Interview a "domain expert" (a classmate or lecturer) for the rules and note their source.

In [ ]:
# TODO(Student): your domain, ≥ 6 rules (one chain), two cases
# TODO: your own rule base (4-6 rules)
MY_RULES = [
    # Rule("M1", ("symptom_a", "symptom_b"), "diagnosis_x", 0.8),
]
# Build the engine from your rules
my_engine = Engine(MY_RULES)
# TODO: test case A
CASE_A = {}        # e.g. {"symptom_a": 1.0, "symptom_b": 0.9}
# TODO: test case B
CASE_B = {}

In [ ]:
#@title ✅ Auto-check for Part C
# Run forward chaining on both cases
wa, ta = my_engine.forward(CASE_A); wb, tb = my_engine.forward(CASE_B)
# Print the rules that fired in each
print("Case A:", [t for t in ta]); print("Case B:", [t for t in tb])
# Does any rule's conclusion feed another rule? (chaining check)
chain = any(r.then in {c for q in MY_RULES for c in q.if_} for r in MY_RULES)
check("≥ 6 rules",                                       lambda: len(MY_RULES) >= 6)
check("at least one chained rule (conclusion used as a condition)", lambda: chain)
check("the two cases lead to different conclusions",     lambda: set(wa) - set(CASE_A) != set(wb) - set(CASE_B) and len(set(wa) - set(CASE_A)) > 0 and len(set(wb) - set(CASE_B)) > 0)

## Part D — A help-desk chatbot (CCMAS lab: *staff/student chat services*)
A **chatbot** is an expert system with a language interface: (1) understand the utterance → facts (intent detection), (2) look up the knowledge base, (3) answer or admit ignorance. We use keyword-overlap intent detection (Week 7's retrieval ideas) and a FAQ knowledge base. **Answers below are placeholders — replace them with your department's real procedures before deployment.**

In [ ]:
#@title 💬 Help-desk bot (TODO: add two FAQ intents)
# FAQ knowledge base for the chatbot
FAQ = {
  "registration": ({"register", "registration", "courses", "course", "add", "drop"}, "Course registration is done on the student portal during the registration window. [Replace with the real procedure.]"),
  "results":      ({"result", "results", "score", "scores", "ca", "grade", "marks"}, "CA scores are shown on the notice board and the portal; see your lecturer for corrections. [Replace.]"),
  "fees":         ({"fees", "fee", "pay", "payment", "school"}, "Fees are paid through the approved payment channels listed on the portal. [Replace.]"),
  # TODO: add "timetable" (exam, timetable, schedule) and "consultation" (lecturer, consultation, meet, office)
}
# Default reply for unknown questions
FALLBACK = "Sorry, I don't know that yet. Please contact the department office."

# Pick the FAQ topic with the most keyword overlap
def intent(text, min_overlap=1):
    # Words in the question
    toks = set(re.findall(r"[a-z]+", text.lower()))
    # Overlap score per topic
    scores = {k: len(toks & kw) for k, (kw, _) in FAQ.items()}
    # Best topic
    best = max(scores, key=scores.get)
    # Accept it only if the overlap is large enough
    return best if scores[best] >= min_overlap else None

# Chatbot reply
def reply(text):
    # Identify the intent
    k = intent(text)
    # Return the answer or the fallback
    return FAQ[k][1] if k else FALLBACK

In [ ]:
#@title ✅ Evaluate the bot on 12 test utterances (accuracy with 95% CI) and chat
# Test questions with the correct intent
TEST = [("How do I register my courses?", "registration"), ("when can I add or drop a course", "registration"), ("Where can I see my CA scores?", "results"),
        ("my result is not on the portal", "results"), ("how do I pay school fees", "fees"), ("What is the fee payment channel?", "fees"),
        ("When is the exam timetable out?", "timetable"), ("exam schedule please", "timetable"), ("How can I meet my lecturer in his office?", "consultation"),
        ("what are the consultation hours", "consultation"), ("Is there a hostel for final-year students?", None), ("tell me a joke", None)]
# Check each prediction
hits = [intent(q) == gold for q, gold in TEST]; k = sum(hits)
# SciPy for the CI
from scipy import stats
# Wilson CI for intent accuracy
ci = stats.binomtest(k, len(hits)).proportion_ci(0.95, method="wilson")
# Print the accuracy
print(f"intent accuracy = {k}/{len(hits)} = {k/len(hits):.2f}  (95% CI [{ci.low:.2f}, {ci.high:.2f}])")
# Check failures
for (q, gold), h in zip(TEST, hits):
    # Print each failure
    if not h: print(f"  ✗ {q!r}: predicted {intent(q)!r}, wanted {gold!r}")
# Try two new questions
for q in ["How do I pay my school fees?", "tell me a joke"]: print("You :", q, "\nBot :", reply(q), "\n")
check("bot answers ≥ 10 of 12 test utterances correctly", lambda: k >= 10)
check("out-of-scope questions fall back gracefully",       lambda: reply("tell me a joke") == FALLBACK)

**Discuss.** Where does this bot fail (synonyms, spelling mistakes → reuse your Week 8 corrector; Yorùbá or Pidgin questions; follow-ups like "and the deadline?")? What does an **LLM-based** assistant add, and what *risks* (hallucinated policies!) does it introduce compared with a rule-based expert system with an explanation facility?

## 📝 Assignment (5 marks, individual, formative)
1. *(2)* Build an expert system of **six or more rules** with certainty factors in a domain you know. Run two cases and show the *how?* explanation for each.
2. *(1)* Compare forward and backward chaining on your rules: how many questions does each ask?
3. *(2)* Extend the help-desk bot to **six or more intents** and report its accuracy with a 95% Wilson CI on **15 new utterances** that you wrote yourself.

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
- Why are expert systems still valuable when machine learning exists? (Hint: explanation, small data, safety, auditing.)
- What is the *knowledge-acquisition bottleneck* and how did you feel it in Part C?
- Why is a certainty factor *not* a probability? Give one situation where treating $0.8$ as "80% likely" would mislead a farmer.

```text
Write your reflection here…
```

## Extensions
- Add **negation** ("no fever") and **numeric** conditions (temperature > 38) to the rule language.
- Use a **MYCIN-style** negative certainty factor in $[-1,1]$ with the full combination rule.
- Connect the bot to the Week 7 TF-IDF retriever as a fallback instead of the fixed message.

In [ ]:
checks_summary()  # how many auto-checks have you passed?